# MoE LLM Pipeline on Colab + Google Drive
This notebook mounts Drive, routes caches and artifacts to Drive, and runs the repo code from Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/LLM_Pretraining_Pipeline'
REPO_ROOT = '/content/LLM_Pretraining_Pipeline'

os.environ['LLM_PROJECT_ROOT'] = REPO_ROOT
os.environ['LLM_ARTIFACT_DIR'] = f'{DRIVE_ROOT}/artifacts'
os.environ['LLM_LOG_DIR'] = f'{DRIVE_ROOT}/artifacts/logs'
os.environ['LLM_RAW_DIR'] = f'{DRIVE_ROOT}/artifacts/data/raw'
os.environ['LLM_PROCESSED_DIR'] = f'{DRIVE_ROOT}/artifacts/data/processed'
os.environ['LLM_TOKENIZER_PATH'] = f'{DRIVE_ROOT}/artifacts/tokenizer/tokenizer.json'
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_ROOT}/hf_cache/datasets'
os.environ['TRANSFORMERS_CACHE'] = f'{DRIVE_ROOT}/hf_cache/transformers'
os.environ['LLM_STREAMING_CACHE_DIR'] = os.environ['HF_DATASETS_CACHE']

for path in [
    DRIVE_ROOT,
    os.environ['LLM_ARTIFACT_DIR'],
    os.environ['LLM_LOG_DIR'],
    os.environ['LLM_RAW_DIR'],
    os.environ['LLM_PROCESSED_DIR'],
    os.environ['HF_HOME'],
    os.environ['HF_DATASETS_CACHE'],
    os.environ['TRANSFORMERS_CACHE'],
]:
    os.makedirs(path, exist_ok=True)

print('Artifacts ->', os.environ['LLM_ARTIFACT_DIR'])
print('Logs ->', os.environ['LLM_LOG_DIR'])
print('HF cache ->', os.environ['HF_DATASETS_CACHE'])

In [ ]:
!rm -rf /content/LLM_Pretraining_Pipeline
!git clone https://github.com/your-user/LLM_Pretraining_Pipeline.git /content/LLM_Pretraining_Pipeline
%cd /content/LLM_Pretraining_Pipeline
!pip install -r requirements.txt

In [ ]:
CONFIG = 'configs/profiles/real_150m_plus_spec.yaml'
!python -u -m src.data.prepare --config {CONFIG} --validate-only

In [ ]:
CONFIG = 'configs/profiles/real_150m_plus_spec.yaml'
!python -u -m src.data.prepare --config {CONFIG}
!python -u -m src.train.pretrain --config {CONFIG}
!python -u -m src.train.sft --config {CONFIG}
!python -u -m src.train.dpo --config {CONFIG}
!python -u -m src.train.grpo --config {CONFIG}
!python -u -m src.eval.evaluate --config {CONFIG} --stage dpo
!python -u -m src.inference.export --config {CONFIG} --stage dpo
!tail -n 80 "$LLM_LOG_DIR/pipeline.log"